# Stage 7: Visualization

Generate animated GIFs showing bone length measurements overlaid on CT images.

**Input**: `labelsBoneLength_v2/*_bone_length_v2.nii.gz`, `images/*.nii.gz`  
**Output**: `gifs_visualization/*_axial_class4_blue.gif`

In [ ]:
# Configuration
TARGET_DIR = "/mnt/c/users/mwild/firebase/perios/levi_data_1.6.26"

# GIF parameters
SLICE_PLANE = 'axial'  # 'axial', 'coronal', 'sagittal'
FPS = 24

# Overlay settings
CLASS4_LABEL = 4  # Bone segment line
DILATION_RADIUS = 3  # Make line more visible
BLUE_ALPHA = 0.9  # Overlay opacity

# Note: Sequential processing is faster than parallel on WSL due to filesystem I/O bottleneck

In [ ]:
import sys
import time
from pathlib import Path
import numpy as np
import nibabel as nib
import imageio
from scipy import ndimage

sys.path.insert(0, str(Path('.').resolve()))
from utils import ensure_dir

In [ ]:
# Setup directories
target = Path(TARGET_DIR)
BONE_LENGTH_DIR = target / "labelsBoneLength_v2"
IMAGES_DIR = target / "images"
OUTPUT_DIR = ensure_dir(target / "gifs_visualization")

print(f"Bone length masks: {BONE_LENGTH_DIR}")
print(f"CT images: {IMAGES_DIR}")
print(f"Output GIFs: {OUTPUT_DIR}")

# Check inputs exist
bone_files = sorted(BONE_LENGTH_DIR.glob("*_bone_length_v2.nii.gz"))
print(f"\nFound {len(bone_files)} bone length files to process")

In [ ]:
def make_gif(ct_path, bone_path, output_path, slice_plane='axial', fps=24,
              class_label=4, dilation_radius=3, alpha=0.9):
    """Create animated GIF with bone measurement line overlaid on CT.
    
    Uses optimized 2D dilation per-slice instead of 3D dilation.
    """
    # Load data
    ct = nib.load(ct_path).get_fdata()
    labels = nib.load(bone_path).get_fdata()
    
    # Check dimension compatibility
    if ct.shape != labels.shape:
        raise ValueError(f"Shape mismatch: CT {ct.shape} vs labels {labels.shape}")
    
    # Scale CT to uint8 grayscale
    ct = ct.astype(np.float32)
    vmin, vmax = np.percentile(ct, (2, 98))
    ct = np.clip(ct, vmin, vmax)
    ct_u8 = ((ct - vmin) / (vmax - vmin + 1e-8) * 255).astype(np.uint8)
    
    # Select slice axis
    axis = {'sagittal': 0, 'coronal': 1, 'axial': 2}[slice_plane]
    
    # Create 2D structuring element for dilation
    struct_2d = ndimage.generate_binary_structure(2, 1)
    struct_2d = ndimage.iterate_structure(struct_2d, dilation_radius)
    
    electric_blue = np.array([0, 180, 255], dtype=np.uint8)
    
    # Generate frames with per-slice 2D dilation
    frames = []
    for i in range(ct_u8.shape[axis]):
        # Get slice
        ct_slice = ct_u8.take(i, axis=axis)
        label_slice = labels.take(i, axis=axis)
        
        # Create RGB from grayscale
        rgb = np.stack([ct_slice, ct_slice, ct_slice], axis=-1)
        
        # 2D dilation on this slice only (FAST)
        mask_2d = label_slice == class_label
        if mask_2d.any():
            mask_2d = ndimage.binary_dilation(mask_2d, structure=struct_2d)
            rgb[mask_2d] = ((1 - alpha) * rgb[mask_2d] + alpha * electric_blue).astype(np.uint8)
        
        frames.append(np.rot90(rgb))
    
    # Save GIF
    imageio.mimsave(str(output_path), frames, fps=fps, loop=0)
    return len(frames)

In [ ]:
# Process all samples sequentially (faster than parallel on WSL)
print("="*70)
print("GENERATING VISUALIZATION GIFs")
print("="*70)
print(f"\nSamples to process: {len(bone_files)}")
print()

start_time = time.time()
results = []
skipped_no_ct = []

for idx, bone_file in enumerate(bone_files, 1):
    sample_name = bone_file.stem.replace('_bone_length_v2', '').replace('.nii', '')
    
    # Find corresponding CT image
    ct_file = IMAGES_DIR / f"{sample_name}.nii.gz"
    if not ct_file.exists():
        ct_file = IMAGES_DIR / f"{sample_name}_0000.nii.gz"
    if not ct_file.exists():
        matches = list(IMAGES_DIR.glob(f"{sample_name}*.nii.gz"))
        if matches:
            ct_file = matches[0]
    
    if not ct_file.exists():
        skipped_no_ct.append(sample_name)
        print(f"[{idx}/{len(bone_files)}] {sample_name}: SKIPPED (no CT)")
        continue
    
    output_file = OUTPUT_DIR / f"{sample_name}_{SLICE_PLANE}_class4_blue.gif"
    
    sample_start = time.time()
    try:
        n_frames = make_gif(
            ct_path=str(ct_file),
            bone_path=str(bone_file),
            output_path=str(output_file),
            slice_plane=SLICE_PLANE,
            fps=FPS,
            class_label=CLASS4_LABEL,
            dilation_radius=DILATION_RADIUS,
            alpha=BLUE_ALPHA
        )
        elapsed = time.time() - sample_start
        results.append((sample_name, 'success', n_frames, None))
        print(f"[{idx}/{len(bone_files)}] {sample_name}: {n_frames} frames ({elapsed:.1f}s)")
    except Exception as e:
        results.append((sample_name, 'error', 0, str(e)))
        print(f"[{idx}/{len(bone_files)}] {sample_name}: ERROR - {e}")

total_time = time.time() - start_time

In [ ]:
# Summary
print("\n" + "="*70)
print("VISUALIZATION COMPLETE")
print("="*70)

successful = [r for r in results if r[1] == 'success']
errors = [r for r in results if r[1] == 'error']

print(f"\nProcessed: {len(successful)} samples")
print(f"Errors: {len(errors)} samples")
print(f"Skipped (no CT): {len(skipped_no_ct)} samples")
print(f"Total time: {total_time:.1f}s ({total_time/max(len(successful),1):.1f}s avg)")

if errors:
    print(f"\nError details:")
    for name, _, _, error in errors:
        print(f"  {name}: {error}")

if skipped_no_ct:
    print(f"\nSkipped samples (no CT image found):")
    for name in skipped_no_ct:
        print(f"  {name}")

print(f"\nOutput directory: {OUTPUT_DIR}")

# List output files
output_files = list(OUTPUT_DIR.glob("*.gif"))
print(f"GIF files created: {len(output_files)}")

if output_files:
    total_size = sum(f.stat().st_size for f in output_files)
    print(f"Total size: {total_size / 1024 / 1024:.1f} MB")